In [2]:

import os

In [3]:

%pwd

'c:\\Users\\Vinay Verma\\Desktop\\code\\Skin-Disease-Detection\\research'

In [2]:

os.chdir("../")

In [5]:

%pwd

'c:\\Users\\Vinay Verma\\Desktop\\code\\Skin-Disease-Detection'

In [3]:

from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path
    final_dataset_dir: Path

In [4]:
from src.cnnClassifier.constants import *
from src.cnnClassifier.utils.common import read_yaml, create_directories

class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=Path(config.root_dir),
            source_URL=config.source_URL,
            local_data_file=Path(config.local_data_file),
            unzip_dir=Path(config.unzip_dir),
            final_dataset_dir=Path(config.final_dataset_dir)
        )

        return data_ingestion_config


In [5]:
import os
import shutil
import zipfile
from pathlib import Path
from dotenv import load_dotenv
from src.cnnClassifier import logger
from src.cnnClassifier.utils.common import get_size


# Load environment variables from the .env file BEFORE importing kaggle
load_dotenv()
import kaggle

class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_file(self):
        """
        Downloads the dataset using the Kaggle API securely via .env credentials.
        """
        if not os.path.exists(self.config.local_data_file):
            logger.info("Authenticating with Kaggle API...")
            
            # Parse the dataset slug from the URL
            # E.g., 'https://.../datasets/tawsifurrahman/covid19-radiography-database' 
            # becomes 'tawsifurrahman/covid19-radiography-database'
            url_parts = self.config.source_URL.split('/')
            dataset_slug = f"{url_parts[-2]}/{url_parts[-1]}"
            
            logger.info(f"Downloading Kaggle dataset: {dataset_slug}")
            
            # Authenticate using the environment variables
            kaggle.api.authenticate()
            
            # Download the zip file to the root directory
            kaggle.api.dataset_download_files(
                dataset_slug, 
                path=self.config.root_dir, 
                unzip=False
            )
            
            # Kaggle saves the file using the dataset name (e.g., covid19-radiography-database.zip)
            # We rename it to our standard 'data.zip' so the rest of the pipeline works seamlessly
            downloaded_zip_name = f"{url_parts[-1]}.zip"
            downloaded_zip_path = os.path.join(self.config.root_dir, downloaded_zip_name)
            
            if os.path.exists(downloaded_zip_path):
                os.rename(downloaded_zip_path, self.config.local_data_file)
            
            logger.info(f"Dataset downloaded successfully to {self.config.local_data_file}")
        else:
            logger.info(f"File already exists. Size: {get_size(Path(self.config.local_data_file))}")  

    def extract_zip_file(self):
        """
        Extracts the raw zip archive into a temporary extraction directory.
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)
        logger.info(f"Extracted zip file into: {unzip_path}")

    def clean_and_restructure_dataset(self):
        """
        Restructure the Skin Disease dataset while preserving the
        Train/Test split and class-wise directory structure.

        Final Structure:

        artifacts/data/
        ├── train/
        │   ├── Acne/
        │   ├── Actinic_Keratosis/
        │   ├── ...
        │   └── Warts/
        │
        └── test/
            ├── Acne/
            ├── Actinic_Keratosis/
            ├── ...
            └── Warts/
        """

        logger.info("Restructuring Skin Disease dataset...")

        extracted_root = Path(self.config.unzip_dir)

        # Find Train and Test folders anywhere inside extracted directory
        train_dir = None
        test_dir = None

        for path in extracted_root.rglob("Train"):
            if path.is_dir():
                train_dir = path
                break

        for path in extracted_root.rglob("Test"):
            if path.is_dir():
                test_dir = path
                break

        if train_dir is None or test_dir is None:
            raise FileNotFoundError(
                "Could not locate Train/Test folders in extracted dataset."
            )

        final_root = Path(self.config.final_dataset_dir)

        final_train = final_root / "train"
        final_test = final_root / "test"

        final_train.mkdir(parents=True, exist_ok=True)
        final_test.mkdir(parents=True, exist_ok=True)

        # -------------------------
        # Copy Train Images
        # -------------------------
        logger.info("Copying training images...")

        for class_dir in train_dir.iterdir():

            if not class_dir.is_dir():
                continue

            destination = final_train / class_dir.name
            destination.mkdir(parents=True, exist_ok=True)

            for img in class_dir.iterdir():

                if img.is_file() and img.suffix.lower() in [
                    ".jpg",
                    ".jpeg",
                    ".png",
                    ".bmp",
                    ".webp",
                ]:
                    shutil.copy2(img, destination / img.name)

        # -------------------------
        # Copy Test Images
        # -------------------------
        logger.info("Copying testing images...")

        for class_dir in test_dir.iterdir():

            if not class_dir.is_dir():
                continue

            destination = final_test / class_dir.name
            destination.mkdir(parents=True, exist_ok=True)

            for img in class_dir.iterdir():

                if img.is_file() and img.suffix.lower() in [
                    ".jpg",
                    ".jpeg",
                    ".png",
                    ".bmp",
                    ".webp",
                ]:
                    shutil.copy2(img, destination / img.name)

        logger.info("Dataset restructuring completed.")

        

In [6]:
STAGE_NAME = "Data Ingestion Stage"

try:
    logger.info(f">>>>>> stage {STAGE_NAME} started <<<<<<")
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
    data_ingestion.clean_and_restructure_dataset()
    logger.info(f">>>>>> stage {STAGE_NAME} completed <<<<<<\n\nx==========x")
except Exception as e:
    logger.exception(e)
    raise e

[2026-07-11 14:33:38,417: INFO: 4224048450: >>>>>> stage Data Ingestion Stage started <<<<<<]
[2026-07-11 14:33:38,421: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-07-11 14:33:38,424: INFO: common: yaml file: params.yaml loaded successfully]
[2026-07-11 14:33:38,426: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-07-11 14:33:38,427: INFO: common: created directory at: artifacts]
[2026-07-11 14:33:38,429: INFO: common: created directory at: artifacts/data_ingestion]
[2026-07-11 14:33:38,430: INFO: 2499362980: Authenticating with Kaggle API...]
[2026-07-11 14:33:38,431: INFO: 2499362980: Downloading Kaggle dataset: pacificrm/skindiseasedataset]
Dataset URL: https://www.kaggle.com/datasets/pacificrm/skindiseasedataset
[2026-07-11 14:36:18,567: INFO: 2499362980: Dataset downloaded successfully to artifacts\data_ingestion\data.zip]
[2026-07-11 14:36:50,470: INFO: 2499362980: Extracted zip file into: artifacts\data_ingestion\raw_extracted]
[2026-07